In [10]:
import asyncio,time

async def delayed_value(value: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return value

c = delayed_value("document-ready", 1)
print("创建协程：", c)
print('await before', time.time())
value = await c
print('await after', time.time())
assert value == "document-ready"

<coroutine object delayed_value at 0x1076e1630>
await before 1791805189.067007
await after 1791805190.068676


In [32]:
a = await delayed_value("A", 1)
b = await delayed_value("B", 2)
print("a, b = ", [a, b])
assert [a, b] == ["A", "B"]

c = await asyncio.gather(
    delayed_value("A", 1),
    delayed_value("B", 2)
)
print("c = ", c)
assert c == ["A", "B"]

a2 = await delayed_value("A", 2)
b2 = await delayed_value("B", 1)
print("a2, b2 = ", [a2, b2])
assert [a2, b2] == ["A", "B"]

c2 = await asyncio.gather(
    delayed_value("A", 2),
    delayed_value("B", 1)
)
print("c2 = ", c2)
assert c2 == ["A", "B"]

a, b =  ['A', 'B']
c =  ['A', 'B']
a2, b2 =  ['A', 'B']
c2 =  ['A', 'B']


In [12]:
import httpx

def handler(request: httpx.Request) -> httpx.Response:
    assert request.url.path == "/status"
    return httpx.Response(200, json={"ready": True})

async def read_status() -> bool:
    transport = httpx.MockTransport(handler)
    async with httpx.AsyncClient(transport=transport) as client:
        response = await client.get(
            "https://training.invalid/status",
            timeout=5.0
        )
        response.raise_for_status()
        payload = response.json()
        return payload["ready"]

co = read_status()
print(co)
read_status_v = await co
print(read_status_v)
assert read_status_v == True


<coroutine object read_status at 0x10713e840>
True


In [47]:
import httpx, pytest
from httpx import HTTPStatusError


def handler503(request: httpx.Request) -> httpx.Response:
    assert request.url.path == "/status"
    return httpx.Response(503, json={"readyd": True})

async def read_status503() -> bool:
    transport = httpx.MockTransport(handler503)
    async with httpx.AsyncClient(transport=transport) as client:
        response = await client.get(
            "https://training.invalid/status",
            timeout=5.0
        )
        response.raise_for_status()
        payload = response.json()
        return payload["ready"]

co503 = read_status503()
with pytest.raises(HTTPStatusError) as exc_info:
    read_status503_v = await co503

assert exc_info.type == HTTPStatusError
assert "503 Service Unavailable" in str(exc_info.value)

In [25]:
import httpx, pytest
import json

def handler_json(request: httpx.Request) -> httpx.Response:
    assert request.url.path == "/status"
    return httpx.Response(200, headers={"Content-Type": "application/json"}, content=b'{"ready": True')

async def read_status_json() -> bool:
    transport = httpx.MockTransport(handler_json)
    async with httpx.AsyncClient(transport=transport) as client:
        response = await client.get(
            "https://training.invalid/status",
            timeout=5.0
        )
        response.raise_for_status()
        payload = response.json()
        return payload["ready"]

co_json = read_status_json()
with pytest.raises(json.JSONDecodeError) as exc_info:
    read_status_json_v = await co_json
assert exc_info.type == json.JSONDecodeError


In [42]:
from typing import Protocol
import pytest

class JsonTransport(Protocol):
    async def get_json(self, url: str, *, timeout: float):
        ...

class FakeTransport:
    def __init__(self, payload: object = None, exception: Exception = None) -> None:
        self.payload = payload
        self.exception = exception
        self.calls: list[tuple[str, float]] = []

    async def get_json(self, url: str, *, timeout: float) -> object:
        self.calls.append((url, timeout))
        if self.exception is not None:
            raise self.exception
        return self.payload

async def use_transport(transport: JsonTransport) -> object:
    return await transport.get_json("https://training.invalid/status", timeout=5.0)

fa = FakeTransport(payload={"a": 1})
r = await use_transport(fa)
assert r == {"a": 1}
assert len(fa.calls) == 1
assert fa.calls == [("https://training.invalid/status", 5.0)]

fa2 = FakeTransport(exception=ValueError("test"))
with pytest.raises(ValueError, match="test") as exc_info:
    r2 = await use_transport(fa2)
assert len(fa.calls) == 1
assert fa.calls == [("https://training.invalid/status", 5.0)]